In [5]:
import cv2
import mediapipe as mp

print('✅ Libraries loaded!')
print(f'OpenCV version: {cv2.__version__}')
print(f'MediaPipe version: {mp.__version__}')

✅ Libraries loaded!
OpenCV version: 5.0.0
MediaPipe version: 0.10.35


In [6]:
import urllib.request

model_url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
model_path = 'hand_landmarker.task'

urllib.request.urlretrieve(model_url, model_path)
print('✅ Model downloaded!')

✅ Model downloaded!


In [7]:
import os
print(os.path.exists('hand_landmarker.task'))
print(f'File size: {os.path.getsize("hand_landmarker.task") / 1024:.1f} KB')

True
File size: 7635.8 KB


In [8]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Configure the hand landmark detector
base_options = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,                     # track one hand only
    min_hand_detection_confidence=0.7,
    running_mode=vision.RunningMode.VIDEO  # we're processing a live video stream
)

detector = vision.HandLandmarker.create_from_options(options)

print('✅ Hand landmark detector ready!')

✅ Hand landmark detector ready!


In [9]:
import time

# Open webcam
cap = cv2.VideoCapture(0)  # 0 = default webcam

if not cap.isOpened():
    print('❌ Could not open webcam')
else:
    print('✅ Webcam opened! Press "q" in the video window to quit.')

frame_timestamp = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # Flip horizontally so it acts like a mirror (feels natural)
    frame = cv2.flip(frame, 1)
    
    # Convert BGR (OpenCV's default) to RGB (MediaPipe's expected format)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Run detection
    frame_timestamp += 1
    result = detector.detect_for_video(mp_image, frame_timestamp)
    
    # Draw landmarks if a hand was detected
    if result.hand_landmarks:
        h, w, _ = frame.shape
        for hand_landmarks in result.hand_landmarks:
            for landmark in hand_landmarks:
                cx, cy = int(landmark.x * w), int(landmark.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
    
    cv2.imshow('Hand Tracking Test', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('Webcam closed.')

✅ Webcam opened! Press "q" in the video window to quit.
Webcam closed.


In [10]:
import csv

# Gesture labels mapped to keyboard keys
GESTURES = {
    '1': 'thumbs_up',
    '2': 'thumbs_down',
    '3': 'peace',
    '4': 'ok',
    '5': 'open_palm',
    '6': 'fist',
    '7': 'love',
    '8': 'point_up'
}

DATA_FILE = 'gesture_data.csv'

# Create the CSV file with headers if it doesn't exist yet
if not os.path.exists(DATA_FILE):
    headers = []
    for i in range(21):  # 21 landmarks
        headers += [f'x{i}', f'y{i}', f'z{i}']
    headers.append('label')
    
    with open(DATA_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
    
    print(f'✅ Created {DATA_FILE} with {len(headers)} columns')
else:
    print(f'{DATA_FILE} already exists — will append new data to it')

✅ Created gesture_data.csv with 64 columns


In [14]:
cap = cv2.VideoCapture(0)
frame_timestamp = 0
current_gesture = None
sample_count = 0

print('=== Gesture Data Collection ===')
print('Press and HOLD a number key (1-8) to record that gesture')
print('Release the key to stop recording')
print('Press "q" to quit\n')
for key, gesture in GESTURES.items():
    print(f'{key} = {gesture}')

with open(DATA_FILE, 'a', newline='') as f:
    writer = csv.writer(f)
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        frame_timestamp = int(time.time() * 1000)
        result = detector.detect_for_video(mp_image, frame_timestamp)
        
        key_pressed = cv2.waitKey(1) & 0xFF
        
        # Check if a valid gesture key is being held
        key_char = chr(key_pressed) if key_pressed != 255 else None
        
        if key_char in GESTURES:
            current_gesture = GESTURES[key_char]
        elif key_pressed == 255:  # no key held
            current_gesture = None
        
        if result.hand_landmarks:
            h, w, _ = frame.shape
            for hand_landmarks in result.hand_landmarks:
                # Draw the landmarks
                for landmark in hand_landmarks:
                    cx, cy = int(landmark.x * w), int(landmark.y * h)
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
                
                # If a gesture key is currently held, save this frame's data
                if current_gesture:
                    row = []
                    for landmark in hand_landmarks:
                        row += [landmark.x, landmark.y, landmark.z]
                    row.append(current_gesture)
                    writer.writerow(row)
                    sample_count += 1
        
        # Display status on screen
        status_text = f'Recording: {current_gesture}' if current_gesture else 'Press a number key...'
        color = (0, 0, 255) if current_gesture else (255, 255, 255)
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.putText(frame, f'Total samples: {sample_count}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.imshow('Gesture Data Collection', frame)
        
        if key_pressed == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print(f'\n✅ Collection session ended. Total samples recorded: {sample_count}')

=== Gesture Data Collection ===
Press and HOLD a number key (1-8) to record that gesture
Release the key to stop recording
Press "q" to quit

1 = thumbs_up
2 = thumbs_down
3 = peace
4 = ok
5 = open_palm
6 = fist
7 = love
8 = point_up

✅ Collection session ended. Total samples recorded: 1294


In [15]:
import pandas as pd

data = pd.read_csv(DATA_FILE)
print(f'Total samples: {len(data)}')
print(f'\nSamples per gesture:')
print(data['label'].value_counts())

Total samples: 1294

Samples per gesture:
label
thumbs_up      260
open_palm      163
love           162
ok             156
fist           150
thumbs_down    142
point_up       140
peace          121
Name: count, dtype: int64
